### Day4. 다중 선형 회귀
당뇨병(diabetes) 데이터를 사용해 다중 선형 회귀를 수행합니다.
- 모든 feature는 표준화(z-score 변환) 되어 있어서 평균 0, 분산 1에 가깝습니다.
- age : 환자의 나이
- sex : 성별
- bmi : 체질량지수
- bp : 평균 혈압
- s1 ~ s6 : 6가지 혈액 검사 결과
- target : 1년 후 당뇨병 진행도 지표


In [6]:
# 데이터 확인
import pandas as pd
pd.set_option('display.width', 120)
path = 'https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/'
df = pd.read_csv(path + 'diabetes.csv')
# print(df.head(3))
# 모델 생성 시 상수항(=절편)을 포함하도록 합니다.
# 종속변수: target
# 독립변수: target을 제외한 모든 변수

# 4-1) 위의 조건에 맞게 OLS모델을 생성하고 summary()를 출력
from statsmodels.api import OLS
# print(df.head(3))
# print(df.shape)
train = df.iloc[:300, :]
test = df.iloc[300:, :]
formula = "target ~ " + " + ".join(df.columns[:-1])
model = OLS.from_formula(formula, train).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 target   R-squared:                       0.515
Model:                            OLS   Adj. R-squared:                  0.498
Method:                 Least Squares   F-statistic:                     30.65
Date:                Fri, 07 Nov 2025   Prob (F-statistic):           5.92e-40
Time:                        14:08:55   Log-Likelihood:                -1622.7
No. Observations:                 300   AIC:                             3267.
Df Residuals:                     289   BIC:                             3308.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    152.3478      3.196     47.671      0.0

In [7]:
# 4-2) 위에서 생성한 모델의 결정계수를 반올림하여 소수점 아래 3자리까지 출력
print(round(model.rsquared, 3))

0.515


In [9]:
# 4-3) 위에서 생성한 모델의 수정된 결정계수를 반올림하여 소수점 아래 3자리까지 출력
print(round(model.rsquared_adj, 3))

0.498


In [13]:
# 4-4) 유의수준 0.05하에서 통계적으로 유의한 독립변수의 개수
print(sum(model.pvalues[1:]<=0.05))

4


In [28]:
# 4-5) 유의수준 0.05하에서 통계적으로 유의한 독립변수만 사용하여 target을 종속변수로 하는 모델을 생성하여
# model2로 저장하고 summary()로 확인
s = model.pvalues[1:]<=0.05
cols = s[s].index
formula2 = "target ~ " + " + ".join(cols)
model2 = OLS.from_formula(formula2, train).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                 target   R-squared:                       0.484
Model:                            OLS   Adj. R-squared:                  0.477
Method:                 Least Squares   F-statistic:                     69.11
Date:                Fri, 07 Nov 2025   Prob (F-statistic):           3.21e-41
Time:                        14:17:03   Log-Likelihood:                -1632.0
No. Observations:                 300   AIC:                             3274.
Df Residuals:                     295   BIC:                             3293.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    152.2306      3.253     46.799      0.0

In [34]:
#4-6) bmi 변수에 대한 회귀 계수를 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
print(round(model2.params['bmi'], 3))

621.373


In [42]:
#4-7) 영향력이 가장 높은 변수 및 변수의 회귀계수를 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
# 중간에 공백을 1개 넣어 2개를 한 줄에 출력해 봅니다.
print(model2.params.idxmax(), round(model2.params.values[2],3), sep=' ')

bmi 621.373


In [44]:
#4-8) 영향력이 가장 낮은 변수 및 변수의 회귀계수를 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
# 중간에 공백을 1개 넣어 2개를 한 줄에 출력해 봅니다.
print(model2.params.idxmin(), round(model2.params.values[1],3), sep=' ')

sex -156.819


In [47]:
#4-9) F통계량을 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
print(round(model2.fvalue, 3))

69.107


In [49]:
#4-10) 독립변수 중 가장 높은 p-value를 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
print(round(model2.pvalues.max(), 3))

0.027


In [62]:
#4-11) 통계적으로 가장 유의한 변수는 무엇인가?
s = model2.pvalues[1:]
for i in range(len(s.index)-1):
    if s.values[i] <= 0.05:
        print(s.index[i])

sex
bmi
bp


In [66]:
#4-12) 통계적으로 가장 유의한 변수의 회귀계수를 구해,
# 반올림하여 소수점 아래 3자리까지 출력합니다.
s = model2.pvalues[1:]
for i in range(len(s.index)-1):
    print(f"{s.index[i]}: 채택" if s.values[i] <= 0.05 else f"{s.index[i]}: 기각")
    print(round(model2.pvalues, 3))

sex: 채택
Intercept    0.000
sex          0.027
bmi          0.000
bp           0.003
s5           0.000
dtype: float64
bmi: 채택
Intercept    0.000
sex          0.027
bmi          0.000
bp           0.003
s5           0.000
dtype: float64
bp: 채택
Intercept    0.000
sex          0.027
bmi          0.000
bp           0.003
s5           0.000
dtype: float64
